# 00 — Setup and canonical split · Deliverable 2

One job: **freeze the split.** This notebook mounts Drive, creates the `deliverable2/` folder tree, downloads
PlantVillage from Kaggle if it isn't already on disk, rebuilds the canonical **70/15/15 stratified, file-level
split with seed 42** — the exact logic from `Step3_1_PlantVillage_Baseline_Models` — and writes
`split_train.csv` / `split_val.csv` / `split_test.csv` plus an **MD5 fingerprint** into `00_split/` on Drive.

Run it **once**. Every later notebook *loads* these CSVs and prints the same `SPLIT_ID`.
**If it ever differs, stop** — every comparable number in the report depends on this split being identical.

In [1]:
# Colab setup: Kaggle credentials from Secrets (with retry) + mount Drive. Harmless when run locally.
import os, time

ON_COLAB = False
try:
    from google.colab import userdata, drive
    ON_COLAB = True
except ModuleNotFoundError:
    print("Not on Colab - using local ~/.kaggle/kaggle.json and a local folder in place of Drive.")

if ON_COLAB:
    ok = False
    for attempt in range(1, 4):
        try:
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
            print(f"Kaggle credentials loaded from Colab Secrets (attempt {attempt}).")
            ok = True
            break
        except Exception as e:
            print(f"  attempt {attempt}: Secrets not ready ({type(e).__name__}); retrying in 3s...")
            time.sleep(3)
    if not ok:
        print("")
        print("Colab Secrets did not respond. Fixes, in order:")
        print("  1) RE-RUN this cell - the timeout is almost always transient.")
        print("  2) Left sidebar -> key icon -> confirm KAGGLE_USERNAME and KAGGLE_KEY exist,")
        print("     each with 'Notebook access' toggled ON, then re-run.")
        print("  3) Still failing? Set os.environ['KAGGLE_USERNAME'/'KAGGLE_KEY'] manually here.")
    drive.mount("/content/drive")

Kaggle credentials loaded from Colab Secrets (attempt 1).
Mounted at /content/drive


## 1. Configuration and the Drive folder tree

The only config block in the notebook — seed, split proportions, expected shape, and paths. The raw images stay
on the **local runtime disk** (fast); only the split CSVs and the fingerprint go to **Drive** (durable, shared).

In [2]:
# ---- the one config block: everything below reads from here ----
from pathlib import Path
from collections import deque
import sys, subprocess, hashlib, random
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

SEED          = 42
SPLIT         = (0.70, 0.15, 0.15)          # train / val / test  (Deliverable 1, Section 7.3)
N_ROWS_EXP    = 54_305                      # PlantVillage colour rendering
N_CLASSES_EXP = 38
SIZES_EXP     = (38_013, 8_146, 8_146)      # what this seed + this logic must produce
FORCE_REBUILD = False                       # True only to deliberately overwrite an already-frozen split
random.seed(SEED); np.random.seed(SEED)

# ---- Drive layout ----
_drive     = Path("/content/drive/MyDrive")
DRIVE_ROOT = _drive if _drive.exists() else Path.home()          # local fallback off Colab
D2         = DRIVE_ROOT / "plant_recognition" / "deliverable2"
SUBDIRS    = ["00_split", "01_notebooks", "02_models", "03_results", "04_figures", "05_report"]
for s in SUBDIRS:
    (D2 / s).mkdir(parents=True, exist_ok=True)

SPLIT_DIR   = D2 / "00_split"
SPLIT_CSV   = {s: SPLIT_DIR / f"split_{s}.csv" for s in ("train", "val", "test")}
FINGERPRINT = SPLIT_DIR / "fingerprint.txt"

# ---- raw images (local disk, shared with the Step 1 / Step 3 notebooks) ----
PV_DIR   = Path.home() / "plant_recognition" / "plantvillage"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff"}

print("deliverable2 :", D2)
print("subfolders   :", ", ".join(SUBDIRS))
print("images       :", PV_DIR)
print(f"config       : SEED={SEED}, SPLIT={SPLIT}, FORCE_REBUILD={FORCE_REBUILD}")

deliverable2 : /content/drive/MyDrive/plant_recognition/deliverable2
subfolders   : 00_split, 01_notebooks, 02_models, 03_results, 04_figures, 05_report
images       : /root/plant_recognition/plantvillage
config       : SEED=42, SPLIT=(0.7, 0.15, 0.15), FORCE_REBUILD=False


## 2. Dataset on disk, and the `color` rendering

Downloads the untouched `abdallahalidev` mirror only if `PV_DIR` is empty, then finds the `color` folder by
breadth-first search (the archive nests it differently on different machines) and fixes the **sorted** class order.

In [3]:
# Download PlantVillage (skips if already on disk), then locate the 'color' rendering
if not PV_DIR.exists() or not any(PV_DIR.iterdir()):
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        from kaggle.api.kaggle_api_extended import KaggleApi
    PV_DIR.mkdir(parents=True, exist_ok=True)
    api = KaggleApi(); api.authenticate()
    api.dataset_download_files("abdallahalidev/plantvillage-dataset",
                               path=str(PV_DIR), unzip=True, quiet=False)
    print("Downloaded to", PV_DIR)
else:
    print("Already present:", PV_DIR)

def locate(root, names, max_depth=4):            # same BFS helper as Step 1 / Step 3.1
    q = deque([(root, 0)])
    while q:
        d, depth = q.popleft()
        if d.is_dir() and d.name.lower() in names:
            return d
        if d.is_dir() and depth < max_depth:
            for c in sorted(d.iterdir()):
                if c.is_dir():
                    q.append((c, depth + 1))
    return None

COLOR_DIR = locate(PV_DIR, {"color"})
assert COLOR_DIR is not None, "Could not find a 'color' folder under PV_DIR - check the download."
class_names = sorted([d.name for d in COLOR_DIR.iterdir() if d.is_dir()])   # sorted = canonical label order
assert len(class_names) == N_CLASSES_EXP, f"expected {N_CLASSES_EXP} classes, found {len(class_names)}"
print("color dir:", COLOR_DIR)
print(f"classes  : {len(class_names)}  ({class_names[0]} ... {class_names[-1]})")

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset


100%|██████████| 2.04G/2.04G [00:20<00:00, 107MB/s] 



Downloaded to /root/plant_recognition/plantvillage
color dir: /root/plant_recognition/plantvillage/plantvillage dataset/color
classes  : 38  (Apple___Apple_scab ... Tomato___healthy)


## 3. Rebuild the canonical 70/15/15 split

`train_test_split` applied **twice with stratification** to the *lists of files* — first peeling off the 15% test
set, then the 15% validation set from the remainder. Disjoint by construction, every class proportionally present
despite the ~36:1 imbalance. Identical code, identical seed, identical result as Step 3.1.

If the CSVs are already on Drive they are **loaded, not rebuilt** — a frozen split is not silently overwritten.

In [4]:
# Rebuild (or reuse) the canonical file-level stratified split - exact Step 3.1 logic
def enumerate_files(color_dir):
    paths, labels = [], []
    for c in class_names:
        for f in sorted((color_dir / c).iterdir()):
            if f.suffix.lower() in IMG_EXTS:
                paths.append(str(f)); labels.append(c)
    return np.array(paths), np.array(labels)

if all(p.exists() for p in SPLIT_CSV.values()) and not FORCE_REBUILD:
    dfs = {s: pd.read_csv(SPLIT_CSV[s]) for s in SPLIT_CSV}
    print("Split already frozen on Drive -> loaded, not rebuilt.  (FORCE_REBUILD=True to overwrite.)")
else:
    paths, labels = enumerate_files(COLOR_DIR)
    print(f"enumerated {len(paths):,} image files")
    test_frac        = SPLIT[2]
    val_frac_of_rest = SPLIT[1] / (SPLIT[0] + SPLIT[1])
    p_tr, p_te,  y_tr, y_te  = train_test_split(
        paths, labels, test_size=test_frac, stratify=labels, random_state=SEED)
    p_tr, p_val, y_tr, y_val = train_test_split(
        p_tr, y_tr, test_size=val_frac_of_rest, stratify=y_tr, random_state=SEED)
    dfs = {s: pd.DataFrame({"filepath": pp, "label": yy}) for s, (pp, yy) in
           {"train": (p_tr, y_tr), "val": (p_val, y_val), "test": (p_te, y_te)}.items()}
    for s, d in dfs.items():
        d.to_csv(SPLIT_CSV[s], index=False)
    print("Built split with train_test_split x2 (stratified) and wrote the CSVs to", SPLIT_DIR)

n_tot = sum(len(d) for d in dfs.values())
print(f"\ntrain / val / test : {len(dfs['train']):,} / {len(dfs['val']):,} / {len(dfs['test']):,}"
      f"  ({len(dfs['train'])/n_tot:.0%} / {len(dfs['val'])/n_tot:.0%} / {len(dfs['test'])/n_tot:.0%})")

Split already frozen on Drive -> loaded, not rebuilt.  (FORCE_REBUILD=True to overwrite.)

train / val / test : 38,013 / 8,146 / 8,146  (70% / 15% / 15%)


## 4. Hard checks, then the fingerprint

Nothing downstream may run unless the split is exactly the canonical one: 54,305 rows, 38 classes present in
every split, no path appearing in two splits. The `SPLIT_ID` is an MD5 over the three CSV digests — one short
string every later notebook prints back.

In [5]:
# ---- hard checks ----
n_tot = sum(len(d) for d in dfs.values())
assert n_tot == N_ROWS_EXP, f"expected {N_ROWS_EXP:,} rows in total, got {n_tot:,}"
sizes = (len(dfs["train"]), len(dfs["val"]), len(dfs["test"]))
assert sizes == SIZES_EXP, f"split sizes {sizes} != canonical {SIZES_EXP} - this is NOT the frozen split"
for s, d in dfs.items():
    assert set(d["label"]) == set(class_names), f"{s} is missing classes"
    assert d["filepath"].is_unique,             f"{s} contains duplicate file paths"
sets = {s: set(d["filepath"]) for s, d in dfs.items()}
assert sets["train"].isdisjoint(sets["val"]) and sets["train"].isdisjoint(sets["test"]) \
       and sets["val"].isdisjoint(sets["test"]), "LEAKAGE: the splits overlap by file path"
assert all(Path(p).exists() for p in dfs["train"]["filepath"].head(50)), \
       "the paths stored in the CSVs do not resolve on this machine"
print(f"OK  {n_tot:,} rows  |  {len(class_names)} classes in every split  |  splits disjoint by file path")

# ---- MD5 fingerprint ----
def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

digests  = {s: md5(SPLIT_CSV[s]) for s in ("train", "val", "test")}
SPLIT_ID = hashlib.md5("".join(digests[s] for s in ("train", "val", "test")).encode()).hexdigest()[:12]

lines  = ["# canonical split fingerprint - plant_recognition / deliverable2",
          f"# seed={SEED}  split={SPLIT}  stratified, file-level",
          f"# rows={n_tot}  classes={len(class_names)}  written={pd.Timestamp.now():%Y-%m-%d %H:%M}"]
lines += [f"split_{s}.csv  rows={len(dfs[s]):>6,}  md5={digests[s]}" for s in ("train", "val", "test")]
lines += [f"SPLIT_ID = {SPLIT_ID}"]
FINGERPRINT.write_text("\n".join(lines) + "\n")

print()
print("\n".join(lines))
print("\nwritten to:", FINGERPRINT)
print("\n>>> Keep SPLIT_ID. Every later notebook prints it - if it ever differs, STOP.")

OK  54,305 rows  |  38 classes in every split  |  splits disjoint by file path

# canonical split fingerprint - plant_recognition / deliverable2
# seed=42  split=(0.7, 0.15, 0.15)  stratified, file-level
# rows=54305  classes=38  written=2026-08-13 09:06
split_train.csv  rows=38,013  md5=7d083aaec519a8d188f5b2349ea27b06
split_val.csv  rows= 8,146  md5=611d00bb5835e052b177d58de357d0af
split_test.csv  rows= 8,146  md5=6ee6c1b7fd1df3942357a5460188848c
SPLIT_ID = 9e33ec57c1ec

written to: /content/drive/MyDrive/plant_recognition/deliverable2/00_split/fingerprint.txt

>>> Keep SPLIT_ID. Every later notebook prints it - if it ever differs, STOP.


---
### Snippet for every later notebook

Paste this instead of any split-building code. It loads the frozen split and re-derives the fingerprint,
so a mismatch fails loudly rather than quietly producing an incomparable number.

```python
SPLIT_DIR = Path("/content/drive/MyDrive/plant_recognition/deliverable2/00_split")
SPLIT_CSV = {s: SPLIT_DIR / f"split_{s}.csv" for s in ("train", "val", "test")}
dfs = {s: pd.read_csv(SPLIT_CSV[s]) for s in SPLIT_CSV}

import hashlib
md5 = lambda p: hashlib.md5(Path(p).read_bytes()).hexdigest()
SPLIT_ID = hashlib.md5("".join(md5(SPLIT_CSV[s]) for s in ("train","val","test")).encode()).hexdigest()[:12]
print("SPLIT_ID =", SPLIT_ID)          # must equal the value in 00_split/fingerprint.txt
```

The MD5 is taken over the CSV bytes, which contain absolute file paths — so later notebooks must **load** these
CSVs from Drive, never rebuild them. Rebuilding under a different runtime path would change the fingerprint even
though the split is the same, and that is a false alarm you don't want to chase at 2 a.m.